In [1]:
from ultralytics import YOLO

# Path to data.yaml created during dataset preparation
data_yaml = "yolo_dataset/data.yaml"

model = YOLO("models/yolo11n.pt")

In [2]:

model.train(
    data=data_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    workers=4,
    name="person_detector_aug",
    augment=True,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.5,
    fliplr=0.5,      # 50% chance horizontal flip
    flipud=0.0,      # vertical flip (if needed)      
    translate=0.1,   # translation
    scale=0.2        # scaling ±20%
)


# Save the final weights in 'runs/detect/person_detector/weights/best.pt'


Ultralytics 8.3.169  Python-3.12.6 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=models/yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=person_detector_aug3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plo

train: Scanning E:\synclabs\edgeface\yolo_dataset\labels\train.cache... 1350 images, 154 backgrounds, 0 corrupt: 100%|██████████| 1350/1350 [00:00<?, ?it/s]


val: Fast image access  (ping: 0.00.0 ms, read: 32.16.5 MB/s, size: 25.6 KB)


val: Scanning E:\synclabs\edgeface\yolo_dataset\labels\val.cache... 168 images, 24 backgrounds, 0 corrupt: 100%|██████████| 168/168 [00:00<?, ?it/s]


Plotting labels to e:\synclabs\edgeface\runs\detect\person_detector_aug3\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)


UnsupportedModelRegistryStoreURIException:  Model registry functionality is unavailable; got unsupported URI 'e:\synclabs\edgeface\runs\mlflow' for model registry data storage. Supported URI schemes are: ['', 'file', 'databricks', 'databricks-uc', 'uc', 'http', 'https', 'postgresql', 'mysql', 'sqlite', 'mssql']. See https://www.mlflow.org/docs/latest/tracking.html#storage for how to run an MLflow server against one of the supported backend storage locations.

In [ ]:
import os
from pathlib import Path
from PIL import Image
import shutil

# ====== CONFIGURATION ======
IMAGES_DIR = "data/ExDark/Images"       # Folder containing Images/<class_name>/*.png
LABELS_DIR = "data/ExDark/Labels"       # Folder containing Labels/<class_name>/*.png.txt
META_FILE = "data/ExDark/imageclasslist.txt"      # File with splits and other metadata
OUTPUT_DIR = "data/ExDark/yolo_dataset" # Output YOLO dataset folder

# All 12 classes in ExDark dataset
CLASSES = [
    "Bicycle", "Boat", "Bottle", "Bus", "Car", "Cat",
    "Chair", "Cup", "Dog", "Motorbike", "People", "Table"
]
CLASS_TO_ID = {cls: idx for idx, cls in enumerate(CLASSES)}

# ============================

# Create output directories
for split in ["train", "val", "test"]:
    os.makedirs(f"{OUTPUT_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUTPUT_DIR}/labels/{split}", exist_ok=True)

# Read the meta file to determine train/val/test split
split_map = {"1": "train", "2": "val", "3": "test"}
meta_info = {}
with open(META_FILE, "r") as f:
    for line in f:
        if line.strip() and not line.startswith("Name"):
            parts = line.split()
            filename = parts[0]  # e.g., 2015_00001.png
            split = split_map[parts[4]]
            meta_info[filename] = split

def convert_bbox(x, y, w, h, img_w, img_h):
    """Convert x,y,w,h (top-left format) to YOLO x_center,y_center,w,h normalized."""
    x_center = (x + w / 2) / img_w
    y_center = (y + h / 2) / img_h
    return x_center, y_center, w / img_w, h / img_h

# Iterate over all label files and convert to YOLO format
for class_name in CLASSES:
    class_label_dir = Path(LABELS_DIR) / class_name
    class_image_dir = Path(IMAGES_DIR) / class_name
    if not class_label_dir.exists():
        print(f"Warning: {class_label_dir} not found.")
        continue

    for label_file in class_label_dir.glob("*.txt"):
        img_name = label_file.stem  # e.g., file_name.png
        image_path = class_image_dir / img_name
        if not image_path.exists():
            continue

        split = meta_info.get(img_name)
        if not split:
            continue  # Skip images not in meta.txt

        # Get image size
        with Image.open(image_path) as img:
            img_w, img_h = img.size

        yolo_lines = []
        with open(label_file, "r") as lf:
            for line in lf:
                if line.startswith("%") or not line.strip():
                    continue
                parts = line.split()
                cls_name = parts[0]
                if cls_name not in CLASS_TO_ID:
                    continue
                cls_id = CLASS_TO_ID[cls_name]
                x, y, w, h = map(float, parts[1:5])
                x_c, y_c, w_n, h_n = convert_bbox(x, y, w, h, img_w, img_h)
                yolo_lines.append(f"{cls_id} {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}")

        # Write YOLO label
        dest_label = Path(OUTPUT_DIR) / "labels" / split / (img_name + ".txt")
        with open(dest_label, "w") as outf:
            outf.write("\n".join(yolo_lines))

        # Copy image
        dest_img = Path(OUTPUT_DIR) / "images" / split / img_name
        shutil.copy(image_path, dest_img)

# Write YOLO data.yaml
data_yaml = f"""train: {OUTPUT_DIR}/images/train
val: {OUTPUT_DIR}/images/val
test: {OUTPUT_DIR}/images/test
nc: {len(CLASSES)}
names: {CLASSES}
"""
with open(Path(OUTPUT_DIR) / "data.yaml", "w") as f:
    f.write(data_yaml)
print("YOLO dataset preparation complete!")

YOLO dataset preparation complete!


In [7]:
import os

# Specify the parent directory containing the subdirectories
parent_directory = "data/ExDark/yolo_dataset/labels" # Replace with your parent directory path

# Iterate through all items in the parent directory
for item in os.listdir(parent_directory):
    item_path = os.path.join(parent_directory, item)

    # Check if the item is a directory
    if os.path.isdir(item_path):
        # Iterate through files in the subdirectory
        for filename in os.listdir(item_path):
            # Check if the file ends with '.png.txt'
            if filename.endswith(".JPG.txt"):
                # Create the new filename by removing '.png'
                new_filename = filename.replace(".JPG.txt", ".txt")
                # Construct the full old and new file paths
                old_filepath = os.path.join(item_path, filename)
                new_filepath = os.path.join(item_path, new_filename)

                # Rename the file
                os.rename(old_filepath, new_filepath)
                print(f"Renamed '{old_filepath}' to '{new_filepath}'")

print("Renaming process complete.")

Renamed 'data/ExDark/yolo_dataset/labels\test\2015_00449.JPG.txt' to 'data/ExDark/yolo_dataset/labels\test\2015_00449.txt'
Renamed 'data/ExDark/yolo_dataset/labels\test\2015_00451.JPG.txt' to 'data/ExDark/yolo_dataset/labels\test\2015_00451.txt'
Renamed 'data/ExDark/yolo_dataset/labels\test\2015_00452.JPG.txt' to 'data/ExDark/yolo_dataset/labels\test\2015_00452.txt'
Renamed 'data/ExDark/yolo_dataset/labels\test\2015_00453.JPG.txt' to 'data/ExDark/yolo_dataset/labels\test\2015_00453.txt'
Renamed 'data/ExDark/yolo_dataset/labels\test\2015_00454.JPG.txt' to 'data/ExDark/yolo_dataset/labels\test\2015_00454.txt'
Renamed 'data/ExDark/yolo_dataset/labels\test\2015_00455.JPG.txt' to 'data/ExDark/yolo_dataset/labels\test\2015_00455.txt'
Renamed 'data/ExDark/yolo_dataset/labels\test\2015_00456.JPG.txt' to 'data/ExDark/yolo_dataset/labels\test\2015_00456.txt'
Renamed 'data/ExDark/yolo_dataset/labels\test\2015_00457.JPG.txt' to 'data/ExDark/yolo_dataset/labels\test\2015_00457.txt'
Renamed 'data/Ex

In [1]:
from ultralytics import YOLO

In [3]:
yolo_pos = YOLO("checkpoints/yolo11n-pose.pt")

100%|██████████| 5.97M/5.97M [00:00<00:00, 21.6MB/s]


In [18]:
yolo_box = YOLO("checkpoints/yolo11n.pt")

100%|██████████| 5.35M/5.35M [00:00<00:00, 13.4MB/s]


In [19]:
print(yolo_box)

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_

In [20]:
print(yolo_pos)

YOLO(
  (model): PoseModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats

In [47]:
import torch
from torchsummary import summary
# or
from torchinfo import summary



In [ ]:
from ultralytics import YOLO

yolo_pose = YOLO("checkpoints/yolo11n-pose.pt")  # your pose model
yolo_box = YOLO("checkpoints/yolo11n.pt")  # your detection


In [ ]:
# Even simpler approach - just modify the forward method
def create_simple_combined_model(detection_model, pose_model):
    """
    Simplest approach - modify one model to output both results
    """
    
    # Use pose model as base
    combined_model = pose_model
    
    # Add detection head as attribute
    combined_model.detection_head = detection_model.model.model[23]
    
    # Store original forward method
    combined_model.original_forward = combined_model.forward
    
    # Storage for intermediate features
    combined_model.features = {}
    
    def hook_fn(name):
        def hook(module, input, output):
            combined_model.features[name] = output
        return hook
    
    # Register hooks on the target layers
    combined_model.model.model[16].register_forward_hook(hook_fn('p3'))
    combined_model.model.model[19].register_forward_hook(hook_fn('p4')) 
    combined_model.model.model[22].register_forward_hook(hook_fn('p5'))
    
    def combined_forward(self, x):
        # Clear features
        self.features.clear()
        
        # Run original forward (pose)
        pose_output = self.original_forward(x)
        
        # Extract features and run detection head
        features = [self.features['p3'], self.features['p4'], self.features['p5']]
        detection_output = self.detection_head(features)
        
        return {
            'detection': detection_output,
            'pose': pose_output
        }
    
    # Replace forward method
    import types
    combined_model.forward = types.MethodType(combined_forward, combined_model)
    
    return combined_model

In [ ]:
import torch
import torch.nn as nn

class HookBasedCombinedYOLO(nn.Module):
    """
    Combined YOLO model using hooks to extract intermediate features
    This avoids the complexity of manually handling skip connections
    """
    def __init__(self, detection_model, pose_model):
        super().__init__()
        
        # Use the pose model as the base (they're identical except the head)
        self.base_model = pose_model
        
        # Store the detection head separately
        self.detection_head = detection_model.model.model[23]
        
        # Storage for intermediate features
        self.intermediate_features = {}
        self.hooks = []
        
        # Register hooks to capture intermediate features
        self._register_feature_hooks()
        
        # Model attributes
        self.nc = getattr(detection_model, 'nc', 80)
        self.names = getattr(detection_model, 'names', None)
    
    def _register_feature_hooks(self):
        """Register forward hooks to capture features at specific layers"""
        
        def create_hook(layer_idx):
            def hook_fn(module, input, output):
                self.intermediate_features[layer_idx] = output
            return hook_fn
        
        # Register hooks at the three feature pyramid levels
        target_layers = [16, 19, 22]  # P3, P4, P5 levels
        
        for i, (layer_idx, layer) in enumerate(self.base_model.model.model.named_children()):
            layer_idx = int(layer_idx)
            if layer_idx in target_layers:
                hook = layer.register_forward_hook(create_hook(layer_idx))
                self.hooks.append(hook)
    
    def forward(self, x):
        """Forward pass through both heads using extracted features"""
        
        # Clear previous features
        self.intermediate_features.clear()
        
        # Forward through the base model (pose model)
        pose_output = self.base_model(x)
        
        # Extract the captured features for detection head
        features = [
            self.intermediate_features[16],  # P3 - 64 channels
            self.intermediate_features[19],  # P4 - 128 channels  
            self.intermediate_features[22],  # P5 - 256 channels
        ]
        
        # Forward through detection head
        detection_output = self.detection_head(features)
        
        return {
            'detection': detection_output,
            'pose': pose_output
        }
    
    def predict_detection_only(self, x):
        """Run only detection, more efficient if pose not needed"""
        self.intermediate_features.clear()
        
        # Forward through base model up to feature extraction points
        with torch.no_grad():
            _ = self.base_model(x)
        
        features = [
            self.intermediate_features[16],
            self.intermediate_features[19], 
            self.intermediate_features[22],
        ]
        
        return self.detection_head(features)
    
    def predict_pose_only(self, x):
        """Run only pose estimation"""
        return self.base_model(x)
    
    def __del__(self):
        """Clean up hooks when object is destroyed"""
        for hook in self.hooks:
            hook.remove()


In [76]:
# Alternative: Extract backbone and manually create FPN
class ManualFeaturePyramid(nn.Module):
    """
    Manually extract backbone and create feature pyramid
    """
    def __init__(self, detection_model, pose_model):
        super().__init__()
        
        # Extract the backbone manually (layers 0-10: encoder)
        self.encoder = nn.Sequential(*list(pose_model.model.model.children())[:11])
        
        # Extract decoder parts manually
        self.decoder_layers = nn.ModuleList(list(pose_model.model.model.children())[11:23])
        
        # Heads
        self.detection_head = detection_model.model.model[23]
        self.pose_head = pose_model.model.model[23]
        
    def forward(self, x):
        # Encoder forward pass
        skip_connections = {}
        
        for i, layer in enumerate(self.encoder):
            x = layer(x)
            # Save skip connection features
            if i in [4, 6, 8]:  # Save features for skip connections
                skip_connections[i] = x
        
        # Decoder forward pass with manual handling
        for i, layer in enumerate(self.decoder_layers):
            layer_idx = i + 11  # Adjust for real layer index
            layer_name = type(layer).__name__
            
            if layer_name == 'Upsample':
                x = layer(x)
            elif layer_name == 'Concat':
                if layer_idx == 12:  # First concat
                    # print(x.shape, skip_connections[6].shape)
                    x = torch.cat([x, skip_connections[6]], dim=1)
                elif layer_idx == 15:  # Second concat
                    print(x.shape, skip_connections[4].shape)
                    x = torch.cat([x, skip_connections[4]], dim=1)
                # Add other concat logic as needed
            else:
                x = layer(x)
                
            # Save intermediate features for heads
            if layer_idx in [16, 19, 22]:
                if layer_idx == 16:
                    p3_features = x
                elif layer_idx == 19:
                    p4_features = x
                elif layer_idx == 22:
                    p5_features = x
        
        # Forward through heads
        features = [p3_features, p4_features, p5_features]
        print(f"Features shapes: {[f.shape for f in features]}")
        detection_output = self.detection_head(features)
        pose_output = self.pose_head(features)
        
        return {
            'detection': detection_output,
            'pose': pose_output
        }



In [77]:
comb_model = ManualFeaturePyramid(yolo_box, yolo_pos)

In [78]:
comb_model(torch.randn(1, 3, 640, 640))

torch.Size([1, 128, 80, 80]) torch.Size([1, 128, 80, 80])


RuntimeError: Given groups=1, weight of size [128, 192, 1, 1], expected input[1, 64, 40, 40] to have 192 channels, but got 64 channels instead

In [ ]:
import torch
import torch.nn as nn

# Practical implementation to combine your YOLO models
def combine_yolo_models(detection_model, pose_model):
    """
    Combine detection and pose models with shared backbone
    Based on your provided architectures
    """
    
    class CombinedYOLOModel(nn.Module):
        def __init__(self, det_model, pose_model):
            super().__init__()
            
            # Extract shared backbone (layers 0-22, everything before the heads)
            # Both models have identical structure from layer 0-22
            self.backbone = nn.Sequential(*list(pose_model.model.model.children())[:-1])
            
            # Extract the different heads
            self.detection_head = det_model.model.model[23]  # Detect layer
            self.pose_head = pose_model.model.model[23]      # Pose layer
            
            # Preserve model attributes
            self.nc = getattr(det_model, 'nc', 80)
            self.names = getattr(det_model, 'names', None)
            
        def forward(self, x):
            # Forward through shared backbone
            # We need to capture the outputs at layers 16, 19, 22 (P3, P4, P5)
            p3_features = None
            p4_features = None
            p5_features = None
            
            for i, layer in enumerate(self.backbone):
                x = layer(x)
                
                # Capture multi-scale features
                if i == 16:  # After layer 16 (C3k2) - P3 features (64 channels)
                    p3_features = x
                elif i == 19:  # After layer 19 (C3k2) - P4 features (128 channels)  
                    p4_features = x
                elif i == 22:  # After layer 22 (C3k2) - P5 features (256 channels)
                    p5_features = x
            
            # Create feature list for heads (P3, P4, P5)
            features = [p3_features, p4_features, p5_features]
            
            # Pass through both heads
            detection_output = self.detection_head(features)
            pose_output = self.pose_head(features)
            
            return {
                'detection': detection_output,
                'pose': pose_output
            }
        
        def predict_detection(self, x):
            """Only run detection head"""
            features = self._extract_features(x)
            return self.detection_head(features)
        
        def predict_pose(self, x):
            """Only run pose head"""
            features = self._extract_features(x)
            return self.pose_head(features)
        
        def _extract_features(self, x):
            """Extract P3, P4, P5 features"""
            features = []
            for i, layer in enumerate(self.backbone):
                x = layer(x)
                if i in [16, 19, 22]:
                    features.append(x)
            return features
    
    return CombinedYOLOModel(detection_model, pose_model)

# Alternative simpler approach - modify existing model
def create_dual_head_model(detection_model, pose_model):
    """
    Create a model with both heads by modifying one of the existing models
    """
    
    # Use pose model as base (they're identical except the head)
    base_model = pose_model
    
    # Add detection head as an additional attribute
    base_model.detection_head = detection_model.model.model[23]
    base_model.original_forward = base_model.forward
    
    def dual_forward(self, x):
        # Get features from backbone (everything except last layer)
        features = []
        temp_x = x
        
        for i, layer in enumerate(self.model.model[:-1]):
            temp_x = layer(temp_x)
            if i in [16, 19, 22]:  # P3, P4, P5 levels
                features.append(temp_x)
        
        # Run both heads
        pose_out = self.model.model[23](features)  # Original pose head
        det_out = self.detection_head(features)     # Added detection head
        
        return {
            'detection': det_out,
            'pose': pose_out
        }
    
    # Replace forward method
    base_model.forward = dual_forward.__get__(base_model, type(base_model))
    
    return base_model

# Verification function
def verify_feature_compatibility(det_model, pose_model):
    """
    Verify that the models can be safely combined
    """
    print("Verifying model compatibility...")
    
    # Check backbone layers (should be identical 0-22)
    det_backbone = list(det_model.model.model.children())[:-1]
    pose_backbone = list(pose_model.model.model.children())[:-1]
    
    print(f"Detection backbone layers: {len(det_backbone)}")
    print(f"Pose backbone layers: {len(pose_backbone)}")
    
    compatible = True
    for i, (det_layer, pose_layer) in enumerate(zip(det_backbone, pose_backbone)):
        if type(det_layer) != type(pose_layer):
            print(f"❌ Layer {i} type mismatch: {type(det_layer).__name__} vs {type(pose_layer).__name__}")
            compatible = False
        else:
            print(f"✅ Layer {i}: {type(det_layer).__name__}")
    
    # Check head differences
    det_head = det_model.model.model[23]
    pose_head = pose_model.model.model[23]
    
    print(f"\nDetection head: {type(det_head).__name__}")
    print(f"Pose head: {type(pose_head).__name__}")
    
    return compatible

# Usage example
def main():
    # Load your models (replace with your actual loading code)
    # detection_model = torch.load('path_to_detection_model.pt')
    # pose_model = torch.load('path_to_pose_model.pt')
    
    # Option 1: Clean combined model
    # combined_model = combine_yolo_models(detection_model, pose_model)
    
    # Option 2: Simpler dual-head modification
    # dual_model = create_dual_head_model(detection_model, pose_model)
    
    # Test the model
    # dummy_input = torch.randn(1, 3, 640, 640)
    # outputs = combined_model(dummy_input)
    # 
    # print("Detection output shape:", outputs['detection'].shape)
    # print("Pose output shape:", outputs['pose'].shape)
    
    pass

# Helper function to extract just the backbone
def extract_shared_backbone(model):
    """
    Extract the shared backbone (layers 0-22) as a standalone model
    """
    backbone = nn.Sequential(*list(model.model.model.children())[:-1])
    
    class BackboneExtractor(nn.Module):
        def __init__(self, backbone_layers):
            super().__init__()
            self.backbone = backbone_layers
            
        def forward(self, x):
            features = []
            for i, layer in enumerate(self.backbone):
                x = layer(x)
                if i in [16, 19, 22]:  # P3, P4, P5
                    features.append(x)
            return features
    
    return BackboneExtractor(backbone)

# Function to analyze output shapes
def analyze_output_shapes(model, input_size=(1, 3, 640, 640)):
    """
    Analyze the output shapes of the combined model
    """
    model.eval()
    with torch.no_grad():
        dummy_input = torch.randn(*input_size)
        outputs = model(dummy_input)
        
        print("Model output analysis:")
        if isinstance(outputs, dict):
            for key, value in outputs.items():
                if isinstance(value, (list, tuple)):
                    print(f"{key}: {[v.shape for v in value]}")
                else:
                    print(f"{key}: {value.shape}")
        else:
            print(f"Output shape: {outputs.shape}")

